# Train a Self-Driving Lab Policy with Unsloth

This notebook uses **Unsloth** for fast quantized training on GPU nodes (e.g. H100). It mirrors `train.ipynb` but loads the model via Unsloth's optimized path with 4-bit quantization and LoRA adapters.

**Model**: Use **Qwen2.5** (compatible with transformers 4.57). Qwen3.5 requires transformers 5.x. Alternatives:
- `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`
- `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`
- `Qwen/Qwen2.5-3B-Instruct`

In [ ]:
# Install Unsloth and training dependencies (run once per session)
# Or use: uv sync --extra train
%pip install -q -U torch transformers datasets trl accelerate bitsandbytes unsloth matplotlib huggingface_hub

# H100: if you get vllm.lora.models error, install compatible vllm first:
# %pip install "vllm==0.8.2"  # requires torch 2.6; may need separate env

# Optional extras used by some reward-scoring paths.
%pip install -q -U sentence-transformers gseapy

In [ ]:
# Unsloth must be imported before trl, transformers, peft
import unsloth  # noqa: F401

from pathlib import Path
import torch

from training_unsloth import make_training_args, run_training
import training_script as base

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

Path("artifacts").mkdir(exist_ok=True)

In [ ]:
args = make_training_args(
    model_id="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",  # Qwen2.5 works with transformers 4.57
    output_dir="artifacts/grpo-unsloth-qwen25-3b",
    dataset_episodes=32,
    rollout_steps=10,
    collection_policy="heuristic",
    reward_backend="local",
    domain_randomise=True,
    num_generations=4,
    max_completion_length=160,
    max_prompt_length=1280,
    max_seq_length=2048,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1.0,
    logging_steps=1,
    save_steps=25,
    trust_remote_code=True,
    dry_run=False,
    seed=42,
)

args

In [ ]:
preview_examples = base.build_prompt_examples(
    dataset_episodes=1,
    rollout_steps=args.rollout_steps,
    collection_policy=args.collection_policy,
    scenario_names=["cardiac_disease_de"],
    seed=args.seed,
    domain_randomise=args.domain_randomise,
)

print(preview_examples[0]["prompt"][:3500])
print("\nReference action:\n", preview_examples[0]["reference_action"])

In [ ]:
# Optional smoke test before a full run.
dry_run_args = make_training_args(**{**vars(args), "dry_run": True})
dry_run_result = run_training(dry_run_args)
len(dry_run_result["examples"])

In [ ]:
from IPython.display import Image, display

train_result = run_training(args)
for name, plot_path in train_result["plot_paths"].items():
    print(name, plot_path)
    display(Image(filename=plot_path))